In [1]:
import pandas as pd
import numpy as np
from otter import Otter
from otter.exceptions import FailedQueryError
from otter import DataFinder
import io
import os
import glob

import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.lines import Line2D
from matplotlib.offsetbox import AnchoredText


from astropy.cosmology import Planck18 as cosmo
import astropy.cosmology.units as cu
from astropy import units as u

In [2]:
db = Otter()

In [3]:
wise_phot = pd.read_csv("data/wise-photometry.csv", index_col=0)
radio_phot = pd.read_csv("ecle-photometry.csv")
meta = pd.read_csv("ecle-metadata.csv")

wise_phot["date"] = wise_phot["date_mjd"]
wise_phot["date_format"] = "mjd"
wise_phot["filter_eff_units"] = wise_phot["filter_eff_unit"]

In [4]:
phot = pd.concat([wise_phot, radio_phot])
phot['bibcode'] = 'private'
phot.to_csv("all-photometry.csv")

In [5]:
otterpath = "private_otter_data"
overwrite = False

if overwrite:
    if os.path.exists(otterpath):
        for file in glob.glob(os.path.join(otterpath, "*")):
            os.remove(file)
        os.rmdir(otterpath)


    private_data = Otter.from_csvs("ecle-metadata.csv", photfile="all-photometry.csv", local_outpath=otterpath)
else:
    private_data = Otter(datadir=otterpath)

In [6]:
private_data = Otter(datadir="private_otter_data")

In [7]:
all_transients = private_data.query(query_private=True)

/home/nfranz/astro-otter/otter/src/otter/io/transient.py:845: UserWarning: Names have the same score! Just using the existing default_name
  warnings.warn(
/home/nfranz/astro-otter/otter/src/otter/io/transient.py:845: UserWarning: Names have the same score! Just using the existing default_name
  warnings.warn(
/home/nfranz/astro-otter/otter/src/otter/io/transient.py:845: UserWarning: Names have the same score! Just using the existing default_name
  warnings.warn(
/home/nfranz/astro-otter/otter/src/otter/io/transient.py:845: UserWarning: Names have the same score! Just using the existing default_name
  warnings.warn(
/home/nfranz/astro-otter/otter/src/otter/io/transient.py:845: UserWarning: Names have the same score! Just using the existing default_name
  warnings.warn(


In [8]:
for t in all_transients:
    try:
        t["photometry"]
    except:
        if t.default_name in meta.name.values:
            print(t)

# plotting the radio and IR light curves

In [9]:
meta.name.to_list()

['2019qiz',
 'SDSS_J0748',
 'SDSS_J0807',
 'SDSS_J0938',
 'SDSS_J0952',
 'SDSS_J1055',
 'SDSS_J1207',
 'SDSS_J1241',
 'SDSS_J1247',
 'SDSS_J1342',
 'SDSS_J1350',
 'SDSS_J1402',
 'SDSS_J1459',
 'SDSS_J1715',
 'SDSS_J2220',
 'AT2017gge',
 'AT2022fpx',
 'AT2022upj',
 'AT2021dms',
 'AT2018bcb',
 'AT2020vdq',
 'AT2021qth',
 'AT2021acak',
 'AT2018dyk']

In [ ]:
radio = private_data.get_phot(
    names=meta.name.to_list(), 
    obs_type="radio", 
    query_private=True, 
    return_type="pandas",
    flux_unit="mJy",
    freq_unit="GHz"
)

fig, axs2d = plt.subplots(4,6, figsize=(20,8), sharex=True)

flat_axs = axs2d.flatten() 

for idx, ((name, data), ax) in enumerate(zip(radio.groupby("name"), flat_axs)):
    
    for label, grp in data.groupby('filter_name'):
        ax.errorbar(
            grp.converted_date,
            grp.converted_flux, 
            yerr=grp.converted_flux_err,
            label=f'{label}',
            linestyle='none',
            # marker='o',
            uplims=grp.upperlimit
        )
    
    if not idx%6:
        ax.set_ylabel('Flux [mJy]')
    
    if idx >= 18:
        ax.set_xlabel('MJD')
        
    ax.set_yscale('log')
            
    classes = db.query(names=name, query_private=True)[0]['classification']
    for c in classes:
        if c['object_class'] in {'AGN', 'TDE'}:
            break
    ecle_class = c['object_class']
        
    at = AnchoredText(f'{name} ({ecle_class})', 'upper left', frameon=False, prop=dict(fontsize=8))
    ax.add_artist(at)
    
    ax.legend(fontsize=6, frameon=True, fancybox=True)
    # ax.set_xscale('log')
    
    fig.savefig("radio-lightcurves.png")

/home/nfranz/astro-otter/otter/src/otter/io/transient.py:845: UserWarning: Names have the same score! Just using the existing default_name
  warnings.warn(
/home/nfranz/astro-otter/otter/src/otter/io/transient.py:632: UserWarning: Unable to apply the source mapping because Cannot set a DataFrame with multiple columns to the single column human_readable_refs
  warnings.warn(f"Unable to apply the source mapping because {exc}")
/home/nfranz/astro-otter/otter/src/otter/io/transient.py:632: UserWarning: Unable to apply the source mapping because 'private'
  warnings.warn(f"Unable to apply the source mapping because {exc}")
/home/nfranz/astro-otter/otter/src/otter/io/transient.py:632: UserWarning: Unable to apply the source mapping because 'private'
  warnings.warn(f"Unable to apply the source mapping because {exc}")
/home/nfranz/astro-otter/otter/src/otter/io/transient.py:632: UserWarning: Unable to apply the source mapping because 'private'
  warnings.warn(f"Unable to apply the source mapp

In [ ]:
def make_sed(tde_name, ax):
    classes = db.query(names=tde_name, query_private=True)[0]['classification']
    for c in classes:
        if c['object_class'] in {'AGN', 'TDE'}:
            break
    ecle_class = c['object_class']

    df = radio[radio.name == tde_name]

    for label, grp in df.groupby('converted_date'):

        ax.errorbar(
            grp.converted_freq, 
            grp.converted_flux,
            fmt='o',
            markersize=3,
            yerr=grp.converted_flux_err,
            label=f'{label:.2f} MJD',
            uplims=grp.upperlimit
        )

        # next(ax._get_lines.prop_cycler)['color']


    ax.set_ylabel('Flux [Jy]')
    ax.set_xlabel('Frequency [GHz]')
    ax.set_yscale('log')

    at = AnchoredText(f'{tde_name} ({ecle_class})', 'center right', frameon=False)
    ax.add_artist(at)

    ax.legend(fontsize=12)
    
ecles_with_good_seds = ['SDSS_J0748','SDSS_J0938','SDSS_J1241']
fig, axs = plt.subplots(1,len(ecles_with_good_seds), figsize=(18,6))
                        
for ii, tde_name in enumerate(ecles_with_good_seds):
    make_sed(tde_name, axs[ii])

fig.savefig("good-radio-seds.png")

In [ ]:
wise = db.get_phot(names=meta.name.tolist(), flux_unit='mag(AB)', wave_unit='nm', return_type='pandas', obs_type='uvoir', query_private=True)

color_map = {'w1':'#ffba49',
             'w2': '#20a39e',
             'w3': '#ef5b5b',
             'w4': 'k'}

filters_to_plot = {'w1', 'w2'}

fig, axs2d = plt.subplots(4,6, figsize=(20,8), sharex=True)

flat_axs = axs2d.flatten() 

idx = 0
for name, data in wise.groupby('name'):
    
    if len(data[data.filter_name.isin(filters_to_plot)]) == 0:
        continue
        
    ax = flat_axs[idx]
    
    for f,d in data.groupby('filter_name'):
        if f not in filters_to_plot: continue
        if idx == 0: 
            label = f
        else:
            label = None
        ax.errorbar(
            d.converted_date, 
            d.converted_flux, 
            yerr=d.converted_flux_err, 
            color=color_map[f], 
            marker='o', 
            linestyle='none', 
            label=label
        )
        
    ax.invert_yaxis()

    if not idx%6:
        ax.set_ylabel('AB Magnitude')
    
    if idx >= 18:
        ax.set_xlabel('MJD')
        
    classes = db.query(names=name, query_private=True)[0]['classification']
    for c in classes:
        if c['object_class'] in {'AGN', 'TDE'}:
            break
    ecle_class = c['object_class']
    
    try:
        discovery_date = db.get_meta(names=name, query_private=True)[0].get_discovery_date()
        ax.axvline(discovery_date.mjd, linestyle='--', color='k')        
        ax.axvline(discovery_date.mjd+365*5, linestyle=':', color='k')        
    except: pass # this is fine it just means we don't have a discovery date
    
    at = AnchoredText(f'{name} ({ecle_class})', 'upper left', frameon=False, prop=dict(fontsize=6))
    ax.add_artist(at)
    
    idx += 1
    
for ax in flat_axs[idx:]:
    ax.remove()
    
fig.legend(fontsize=18, bbox_to_anchor=(0.9,0.25), ncol=2, frameon=True, edgecolor='k', framealpha=1, fancybox=False);
fig.savefig("wise-lightcurves.png")

# Some more complex plots

In [ ]:

import warnings
warnings.filterwarnings("ignore")

plotted_with_c = []

def lightcurve_plot(band="S", n_ecle=100):
    
    fig, ax = plt.subplots(figsize=(12,6))
    cmap = mpl.colormaps['plasma']

    ecle_names = set(meta.name.unique())
    
    nondet_color = "#587B7F"
    det_color = "#92140C"
    
    # first figure out how many ECLEs have radio data with no upperlimits
    colors = cmap(np.linspace(0, 1, n_ecle))
    i = 0

    cbar_labels = []

    for t in all_transients:
        try:
            phot = t.clean_photometry(flux_unit="mJy", obs_type="radio")
        except FailedQueryError:
            continue # this just means no radio photometry

        # get the luminosity
        z = t.get_redshift() * cu.redshift
        lum_dist = z.to(u.Mpc, equivalencies=cu.redshift_distance(cosmo, kind="luminosity"))
        phot["lum_nu"] = [4*np.pi*lum_dist**2 * (val * u.mJy) for val in phot.converted_flux]
        phot["lum"] = [(val*eff*u.GHz).to(u.erg/u.s).value for val, eff in zip(phot.lum_nu, phot.converted_freq)]

        # get the discovery date
        try:
            disc_date = t.get_discovery_date().mjd
        except:
            continue

        aliases = [n["value"] for n in t["name"]["alias"]]
        isECLE = any(a in ecle_names for a in aliases)

        if isECLE:
            for a in aliases:
                if a in ecle_names:
                    metadata = meta[a == meta.name]
                    break
            classification = metadata.classification.values[0]

            # All of the ECLEs have S-band data
            s = phot[phot.filter_name == band]
            cbar_labels.append(metadata.name.values[0].replace("_", ""))
            
            c = phot[(phot.filter_name == "C") * (~phot.upperlimit.astype(bool))]
            if np.all(s.upperlimit) and len(c) > 0:
                s = pd.concat([s,c]) # they are close enough in frequency...
                plotted_with_c.append(name)
                
            if np.all(s.upperlimit):
                color = nondet_color
                zorder = 50
                alpha = 1
                                
            else:
                color =  det_color #colors[i]
                zorder = 100
                alpha = 1
                i += 1
            

        else:
            s = phot[phot.filter_name == band]
            if len(s) == 0: continue
            classification = t.get_classification()[0]
            color = 'grey'
            alpha = 0.1
            zorder = 1

        if classification == "TDE":
            symb = "o"
        elif classification == "AGN":
            symb = "*"
        else:
            symb = "s"

        x = s.converted_date.values - disc_date
        ordered_idx = np.argsort(x)
        x = x[ordered_idx]
        y = s.lum.values[ordered_idx]
        
    
        ax.plot(x, y, color=color, marker=symb, alpha=alpha, zorder=zorder)

        uplims = s[s.upperlimit]
        if len(uplims) == 0: continue
        for x, y in zip(uplims.converted_date.values - disc_date, uplims.lum.values):
            ax.annotate("", xy=(x, y/3), xytext=(x, y), arrowprops=dict(arrowstyle='->',color=color,alpha=alpha))

    custom_legend_params = dict(
        color = "k",
        lw = 4,
        markersize = 5,
        linestyle = "none"
    )

    custom_lines = [
        Line2D([0], [0], marker="*", **custom_legend_params),
        Line2D([0], [0], marker="o", **custom_legend_params),
        Line2D([0], [0], marker="s", **custom_legend_params),
        Line2D([0], [0], marker="o", alpha=0.1, **custom_legend_params),
        Line2D([0], [0], marker="o", alpha=1, color=det_color, lw=4, markersize=5, linestyle="none"),
        Line2D([0], [0], marker="o", alpha=1, color=nondet_color, lw=4, markersize=5, linestyle="none")
        
    ]

    custom_labels = ["AGN", "TDE", "Unclear", "TDE Sample", "ECLE\n(Radio Bright)", "ECLE\n(Radio Dim)"]

    ax.set_yscale("log")

    ax.legend(
        custom_lines, 
        custom_labels, 
        # bbox_to_anchor=(1.1,1), 
        ncols = 1, #len(custom_lines)//2, 
        fontsize=10
    )

    #norm = mpl.colors.BoundaryNorm(range(len(colors)), cmap.N)
    #cbar = fig.colorbar(mpl.cm.ScalarMappable(cmap=cmap, norm=norm), ax=ax, ticks=list(range(len(cbar_labels))))
    #cbar.ax.set_yticklabels(cbar_labels, fontsize=10);

    ax.set_ylabel("Luminosity [erg/s]")
    ax.set_xlabel("Date - Discovery Date [days]")

    fig.savefig("ecle-radio-lightcurves.png")
    
    print(f"{i+1} ECLEs have radio data with no upperlimits")
    print(f"{plotted_with_c} were plotted with s and c band")
    
lightcurve_plot("S", n_ecle=18)

In [ ]:
fig, ax = plt.subplots()
cmap = mpl.colormaps['plasma']

ecle_names = set(meta.name.unique())
n_ecle = len(meta)
colors = cmap(np.linspace(0, 1, n_ecle))
i = 0

cbar_labels = []

for t in all_transients:
    try:
        phot = t.clean_photometry(flux_unit="mJy", obs_type="radio")
    except FailedQueryError:
        continue # this just means no radio photometry
    
    # query wise
    coord = t.get_skycoord()
    data_finder = DataFinder(
        coord.ra.value, 
        coord.dec.value, 
        coord.ra.unit,
        coord.dec.unit,
        name = t.default_name
    )
    wise_phot = data_finder.query_wise(overwrite=False)
    
    if len(wise_phot) == 0:
        print(t.default_name, "has radio photometry but no good wise photometry!")
        continue
    
    keep_keys = ["filter", "flux", "flux_err", "date_mjd", "upperlimit"]
    w1 = wise_phot[wise_phot["filter"] == "w1"][keep_keys]
    w2 = wise_phot[wise_phot["filter"] == "w2"][keep_keys]
    
    w1_w2 = w1.merge(w2, on="date_mjd", suffixes=("_w1", "_w2")).dropna()

    wise_color = w1_w2.flux_w1 - w1_w2.flux_w2
    wise_max = np.argmin(wise_color) # because mags are backwards
    wise_uplim = w1_w2.upperlimit_w1.values[wise_max] or w1_w2.upperlimit_w2.values[wise_max]
    
    # get the luminosity
    z = t.get_redshift() * cu.redshift
    lum_dist = z.to(u.Mpc, equivalencies=cu.redshift_distance(cosmo, kind="luminosity"))
    phot["lum_nu"] = [4*np.pi*lum_dist**2 * (val * u.mJy) for val in phot.converted_flux]
    phot["lum"] = [(val*eff*u.GHz).to(u.erg/u.s).value for val, eff in zip(phot.lum_nu, phot.converted_freq)]
    
    # get the discovery date
    try:
        disc_date = t.get_discovery_date().mjd
    except:
        continue
    
    aliases = [n["value"] for n in t["name"]["alias"]]
    isECLE = any(a in ecle_names for a in aliases)
    
    if isECLE:
        for a in aliases:
            if a in ecle_names:
                metadata = meta[a == meta.name]
                break
        classification = metadata.classification.values[0]
        
        color = colors[i]
        i += 1
        
        # All of the ECLEs have S-band data
        s = phot[phot.filter_name == "S"]
        cbar_labels.append(metadata.name.values[0].replace("_", ""))
        
        alpha = 1
        
    else:
        s = phot[phot.filter_name == "S"]
        if len(s) == 0: continue
        classification = t.get_classification()[0]
        color = 'grey'
        alpha = 0.1
    
    if classification == "TDE":
        symb = "o"
    elif classification == "AGN":
        symb = "*"
    else:
        symb = "s"
    
    max_idx = np.argmax(s.lum.values)
    y = s.lum.values[max_idx]
    
    x = wise_color.values[wise_max]
    
    ax.plot(x, y, color=color, marker=symb, alpha=alpha)
    
    if s.upperlimit.values[max_idx]:
        ax.annotate("", xy=(x, y/3), xytext=(x, y), arrowprops=dict(arrowstyle='->',color=color,alpha=alpha))

    if wise_uplim:
        ax.annotate("", xy=(y, x/3), xytext=(y, x), arrowprops=dict(arrowstyle='->',color=color,alpha=alpha))

custom_legend_params = dict(
    color = "k",
    lw = 4,
    markersize = 5,
    linestyle = "none"
)
        
custom_lines = [
    Line2D([0], [0], marker="*", **custom_legend_params),
    Line2D([0], [0], marker="o", **custom_legend_params),
    Line2D([0], [0], marker="s", **custom_legend_params),
    Line2D([0], [0], marker="o", alpha=0.1, **custom_legend_params)
]

custom_labels = ["AGN", "TDE", "Unclear", "TDE Sample"]
        
ax.set_yscale("log")

ax.legend(custom_lines, custom_labels, bbox_to_anchor=(1,1.1), ncols=len(custom_lines), fontsize=10)

norm = mpl.colors.BoundaryNorm(range(len(colors)), cmap.N)
cbar = fig.colorbar(mpl.cm.ScalarMappable(cmap=cmap, norm=norm), ax=ax, ticks=list(range(len(cbar_labels))))
cbar.ax.set_yticklabels(cbar_labels, fontsize=10);

ax.set_ylabel("Radio Luminosity [erg/s]")
ax.set_xlabel("NEOWISE Color (W1 - W2)")

fig.savefig("ecle-radio-vs-wise.png")

In [ ]:
fig, ax = plt.subplots()
cmap = mpl.colormaps['plasma']

ecle_names = set(meta.name.unique())
n_ecle = len(meta)
colors = cmap(np.linspace(0, 1, n_ecle))
i = 0

wise_filter = "w2"

cbar_labels = []

for t in all_transients:
    try:
        phot = t.clean_photometry(flux_unit="mJy", obs_type="radio")
    except FailedQueryError:
        continue # this just means no radio photometry
    
    # query wise
    coord = t.get_skycoord()
    data_finder = DataFinder(
        coord.ra.value, 
        coord.dec.value, 
        coord.ra.unit,
        coord.dec.unit,
        name = t.default_name
    )
    wise_phot = data_finder.query_wise(overwrite=False)
    
    if len(wise_phot) == 0:
        print(t.default_name, "has radio photometry but no good wise photometry!")
        continue
    
    # get the luminosity
    z = t.get_redshift() * cu.redshift
    lum_dist = z.to(u.Mpc, equivalencies=cu.redshift_distance(cosmo, kind="luminosity"))
    phot["lum_nu"] = [4*np.pi*lum_dist**2 * (val * u.mJy) for val in phot.converted_flux]
    phot["lum"] = [(val*eff*u.GHz).to(u.erg/u.s).value for val, eff in zip(phot.lum_nu, phot.converted_freq)]
    
    # get the wise filter absolute magnitude
    keep_keys = ["filter", "flux", "flux_err", "date_mjd", "upperlimit"]
    w = wise_phot[wise_phot["filter"] == wise_filter][keep_keys]
    w["abs_mag"] = [val - 5*np.log10(lum_dist.to(u.pc).value) + 5 for val in w.flux]
    wise_max = np.argmin(w.abs_mag)
    
    # get the discovery date
    try:
        disc_date = t.get_discovery_date().mjd
    except:
        continue
    
    aliases = [n["value"] for n in t["name"]["alias"]]
    isECLE = any(a in ecle_names for a in aliases)
    
    if isECLE:
        for a in aliases:
            if a in ecle_names:
                metadata = meta[a == meta.name]
                break
        classification = metadata.classification.values[0]
        
        color = colors[i]
        i += 1
        
        # All of the ECLEs have S-band data
        s = phot[phot.filter_name == "S"]
        cbar_labels.append(metadata.name.values[0].replace("_", ""))
        
        alpha = 1
        
    else:
        s = phot[phot.filter_name == "S"]
        if len(s) == 0: continue
        classification = t.get_classification()[0]
        color = 'grey'
        alpha = 0.1
    
    if classification == "TDE":
        symb = "o"
    elif classification == "AGN":
        symb = "*"
    else:
        symb = "s"
    
    max_idx = np.argmax(s.lum.values)
    y = s.lum.values[max_idx]
    
    x = w.abs_mag.values[wise_max]
    
    ax.plot(x, y, color=color, marker=symb, alpha=alpha)
    
    if s.upperlimit.values[max_idx]:
        ax.annotate("", xy=(x, y/3), xytext=(x, y), arrowprops=dict(arrowstyle='->',color=color,alpha=alpha))

    if wise_uplim:
        ax.annotate("", xy=(y, x/3), xytext=(y, x), arrowprops=dict(arrowstyle='->',color=color,alpha=alpha))

custom_legend_params = dict(
    color = "k",
    lw = 4,
    markersize = 5,
    linestyle = "none"
)
        
custom_lines = [
    Line2D([0], [0], marker="*", **custom_legend_params),
    Line2D([0], [0], marker="o", **custom_legend_params),
    Line2D([0], [0], marker="s", **custom_legend_params),
    Line2D([0], [0], marker="o", alpha=0.1, **custom_legend_params)
]

custom_labels = ["AGN", "TDE", "Unclear", "TDE Sample"]
        
ax.set_yscale("log")

ax.legend(custom_lines, custom_labels, bbox_to_anchor=(1,1.1), ncols=len(custom_lines), fontsize=10)

norm = mpl.colors.BoundaryNorm(range(len(colors)), cmap.N)
cbar = fig.colorbar(mpl.cm.ScalarMappable(cmap=cmap, norm=norm), ax=ax, ticks=list(range(len(cbar_labels))))
cbar.ax.set_yticklabels(cbar_labels, fontsize=10);

ax.set_ylabel("Radio Luminosity [erg/s]")
ax.set_xlabel(f"NEOWISE {wise_filter.upper()}")

fig.gca().invert_xaxis()

fig.savefig(f"ecle-radio-vs-wise-{wise_filter}-max.png")

In [ ]:
fig, ax = plt.subplots()
cmap = mpl.colormaps['plasma']

ecle_names = set(meta.name.unique())
n_ecle = len(meta)
colors = cmap(np.linspace(0, 1, n_ecle))
i = 0

wise_filter = "w2"

cbar_labels = []

for t in all_transients:
    try:
        phot = t.clean_photometry(flux_unit="mJy", obs_type="radio")
    except FailedQueryError:
        # print(f"Something with the query for {t.default_name} failed!")
        continue # this just means no radio photometry
        
    aliases = [n["value"] for n in t["name"]["alias"]]
    isECLE = any(a in ecle_names for a in aliases)
    
    # query wise
    coord = t.get_skycoord()
    data_finder = DataFinder(
        coord.ra.value, 
        coord.dec.value, 
        coord.ra.unit,
        coord.dec.unit,
        name = t.default_name
    )
    wise_phot = data_finder.query_wise(overwrite=False)
    
    if len(wise_phot) == 0:
        print(t.default_name, "has radio photometry but no good wise photometry!")
        continue
    
    # get the luminosity
    z = t.get_redshift() * cu.redshift
    lum_dist = z.to(u.Mpc, equivalencies=cu.redshift_distance(cosmo, kind="luminosity"))
    phot["lum_nu"] = [4*np.pi*lum_dist**2 * (val * u.mJy) for val in phot.converted_flux]
    phot["lum"] = [(val*eff*u.GHz).to(u.erg/u.s).value for val, eff in zip(phot.lum_nu, phot.converted_freq)]
    
    # get the wise filter absolute magnitude
    keep_keys = ["filter", "flux", "flux_err", "date_mjd", "upperlimit"]
    w = wise_phot[wise_phot["filter"] == wise_filter][keep_keys]
    w["abs_mag"] = [val - 5*np.log10(lum_dist.to(u.pc).value) + 5 for val in w.flux]
    
    try:
        disc_date = t.get_discovery_date().mjd
    except:
        continue
    
    echo_data = []
    dust_echo_time = 5*365 # starting time range
    exc = {"AT2020vdq"}
    while len(echo_data) == 0:
        echo_data = w[(w.date_mjd > disc_date) * (w.date_mjd < disc_date+dust_echo_time)]
        dust_echo_time += 365 # increase the echo time range by one year for some of these SDSS ECLEs
        if (not isECLE or t.default_name in exc) and dust_echo_time >= 10*365: 
            break
    if len(echo_data) == 0: continue
    
    wise_max = echo_data.abs_mag.min() # take the maximum just inside the dust echo range
    wise_mean = w.abs_mag.median() # take the average over all time to try to get the baseline value
    wise_echo = np.abs(wise_max - wise_mean)
    
    if isECLE:
        for a in aliases:
            if a in ecle_names:
                metadata = meta[a == meta.name]
                break
        classification = metadata.classification.values[0]
        
        color = colors[i]
        i += 1
        
        # All of the ECLEs have S-band data
        s = phot[phot.filter_name == "S"]
        cbar_labels.append(metadata.name.values[0].replace("_", ""))
        
        alpha = 1
                
    else:
        s = phot[phot.filter_name == "S"]
        if len(s) == 0: continue
        classification = t.get_classification()[0]
        color = 'grey'
        alpha = 0.1
        
    if classification == "TDE":
        symb = "o"
    elif classification == "AGN":
        symb = "*"
    else:
        symb = "s"
    
    
    max_idx = np.argmax(s.lum.values)
    y = s.lum.values[max_idx]
    
    x = wise_echo
    
    ax.plot(x, y, color=color, marker=symb, alpha=alpha)
    
    if s.upperlimit.values[max_idx]:
        ax.annotate("", xy=(x, y/3), xytext=(x, y), arrowprops=dict(arrowstyle='->',color=color,alpha=alpha))

    if wise_uplim:
        ax.annotate("", xy=(y, x/3), xytext=(x,y), arrowprops=dict(arrowstyle='->',color=color,alpha=alpha))
    
    if isECLE:
        ax.annotate(t.default_name.replace("_", ""), xy=(x-0.05, y+y/5), color=color, fontsize=6)

custom_legend_params = dict(
    color = "k",
    lw = 4,
    markersize = 5,
    linestyle = "none"
)
        
custom_lines = [
    Line2D([0], [0], marker="*", **custom_legend_params),
    Line2D([0], [0], marker="o", **custom_legend_params),
    Line2D([0], [0], marker="s", **custom_legend_params),
    Line2D([0], [0], marker="o", alpha=0.1, **custom_legend_params)
]

custom_labels = ["AGN", "TDE", "Unclear", "TDE Sample (w/ WISE DATA)"]
        
ax.set_yscale("log")

ax.legend(custom_lines, custom_labels, bbox_to_anchor=(1,1.1), ncols=len(custom_lines), fontsize=10)

#norm = mpl.colors.BoundaryNorm(range(len(colors)), cmap.N)
#cbar = fig.colorbar(mpl.cm.ScalarMappable(cmap=cmap, norm=norm), ax=ax, ticks=list(range(len(cbar_labels))))
#cbar.ax.set_yticklabels(cbar_labels, fontsize=10);

ax.set_ylabel("Peak Radio Luminosity [erg/s]")
ax.set_xlabel(f"NEOWISE {wise_filter.upper()} IR Echo Brightness")

fig.savefig(f"ecle-radio-vs-wise-{wise_filter}-echo.png")